To build AI conversational memory
a framework for building stateful, multi-step applications powered by LLMs(GPT - claude)
build flows like converations and processes that involved(decisions, memory, loops) using Graph based structure
LLMs are stateless, they don't have memory
it has Graph state, it means, informations travel from one node to the next
it has "user input", "tools output", "external data"
Nodes: are user defined functions that perform a sprecefic task
Edges: define how data flows from one node to the next
State: represent shared and updated data as the graph progresses, include all informations:user input - assistant response - history - retrieved docs - tool output , ...
schema: define structure and rules to find what that state should like
Start -> Question -> answer -> summarize discussion -> End

In [1]:
from langgraph.graph import START, END, StateGraph, add_messages
from typing_extensions import TypedDict #define schema of graph
from langchain_openai.chat_models import ChatOpenAI
from langchain_core.runnables import Runnable #check graphs are runnable
from langchain_core.messages import AIMessage, HumanMessage, BaseMessage #used for type-checking
from collections.abc import Sequence #for type
from typing import Literal, Annotated 

C:\Users\babaei.nima\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
appended_messages = add_messages([HumanMessage("Hi, this is Nima"), AIMessage("Hi Nima, how can I assist you?")],
                              [HumanMessage("could you sammarize every thing I ask you?")])

# add_messages: are called Reducer messages also, define how to combine the current state with the new incoming data in a structured way

define state

In [ ]:
# Define over writing messages State

#class State(TypedDict):
    #messages: Sequence[BaseMessage] 


#Type Hint, specifies the expected data type for the value associated with the 'message' key
#In this function by default, the State values are relplaced after each node.
#To save valuse in memory, we want new values are appended to the existing state.

In [3]:
# Define appended messages State
class State(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

In [ ]:
#state = State(messages= [HumanMessage("could you tell me how brain work?")])
#state
#state['messages'][0].pretty_print()

define the nodes


In [4]:
llm = ChatOpenAI(
    base_url="http://localhost:1234/v1",
    api_key="not-needed",
    model="qwen2.5-3b-instruct",
    temperature=0,
    max_completion_tokens=250
)

In [ ]:
#response = llm.invoke(state["messages"])
#response
#response.pretty_print()

In [5]:
# ask_question node
def ask_question(state: State) -> State:

    print(f"\n------->  ENTERING ask_question:")
    for i in state["messages"]:
        i.pretty_print()

    question = "what is your question?"
    print(question)

    return State(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
#ask_question(State(messages=[]))

In [6]:
#chatbot node
def chatbot(state: State) -> State:  #this function is good when the number of nodes increasing
    
    print(f"\n------->  ENTERING chatbot:")
    for i in state["messages"]:
        i.pretty_print()

    response = llm.invoke(state["messages"])
    response.pretty_print()

    return State(messages= [response])

In [ ]:
#chatbot(state=state)

In [7]:
# ask_another_question node 
def ask_another_question(state: State) -> State:

    print(f"\n------->  ENTERING ask_another_question:")
    for i in state["messages"]:
        i.pretty_print()

    question = "would you like to ask another question (yes/no)?"
    print(question)

    return State(messages= [AIMessage(question), HumanMessage(input())])

In [ ]:
#ask_another_question(State(messages=[]))

In [8]:
# Define routing function, conditional edges
def routing_function(state: State) -> Literal["ask_question", "__end__"]: #or can put str instead of Literal[...]

    if state["messages"][-1].content == "yes": #check last message to append new message, change from 0 to -1
        return "ask_question"
    else:
        return "__end__"

the example above graph:    
Start -> chatbot -> end  with State defined

Define the Graph

stategraph: a graph whose nodes communicated by reading and writing to a shared state

In [9]:
graph = StateGraph(State)

In [10]:
graph.add_node("ask_question", ask_question)
graph.add_node("chatbot", chatbot)
graph.add_node("ask_another_question", ask_another_question)

graph.add_edge(START, "ask_question")
graph.add_edge("ask_question", "chatbot")
graph.add_edge("chatbot", "ask_another_question")
graph.add_conditional_edges(source= "ask_another_question", 
                            path= routing_function)
                            #path_map= {"True": "ask_question", "False": "__end__"}) after replacing Literal with str

In [11]:
graph_compiled = graph.compile() 
# perform checks to ensure the graph is connected correctly
# converts state_graph into a compiled_state_graph (a runnable)
# isinstance(graph_compiled, Runnable) #checking being runnabe
#graph_compiled.invoke(chatbot(state))   needs internet

In [ ]:
# type chacking
# %mypy
# a: Literal[1,2,3] #means 'a' can only be 1,2,3

In [12]:
graph_compiled.invoke(State(messages=[]))


------->  ENTERING ask_question:
what is your question?

------->  ENTERING chatbot:
================================== Ai Message ==================================

what is your question?
================================ Human Message =================================

please give me the list of middle east countries
================================== Ai Message ==================================

Certainly! Here is a list of Middle Eastern countries:

1. **Arab Republic of Egypt**
2. **Syrian Arab Republic**
3. **Lebanon**
4. **Israel**
5. **Palestine (recognized by some states)**
6. **Jordan**
7. **Iraq**
8. **Kuwait**
9. **Saudi Arabia**
10. **United Arab Emirates (UAE)**
11. **Oman**
12. **Qatar**
13. **Bahrain**
14. **Yemen**

Please note that Palestine is not recognized as an independent state by all countries, and its status remains a subject of international debate and conflict.

------->  ENTERING ask_another_question:
================================== Ai Message =========

{'messages': [AIMessage(content='what is your question?', additional_kwargs={}, response_metadata={}, id='a02bd6de-06cd-4823-b26a-115fadb88fc3', tool_calls=[], invalid_tool_calls=[]),
  HumanMessage(content='please give me the list of middle east countries', additional_kwargs={}, response_metadata={}, id='7f3d1660-234c-440e-a449-34005a0b4029'),
  AIMessage(content='Certainly! Here is a list of Middle Eastern countries:\n\n1. **Arab Republic of Egypt**\n2. **Syrian Arab Republic**\n3. **Lebanon**\n4. **Israel**\n5. **Palestine (recognized by some states)**\n6. **Jordan**\n7. **Iraq**\n8. **Kuwait**\n9. **Saudi Arabia**\n10. **United Arab Emirates (UAE)**\n11. **Oman**\n12. **Qatar**\n13. **Bahrain**\n14. **Yemen**\n\nPlease note that Palestine is not recognized as an independent state by all countries, and its status remains a subject of international debate and conflict.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens':